# Export xTB Descriptor Structures

Build the xTB descriptor input tree from the released `BorylXAT-DB.db` and `Data/TS/Borane_all.csv`.

The exported species are the four non-TS structures used by the DFT descriptor model:

- `B_N_r`: open-shell LB-B radical complex, `uhf=1`
- `B_N_p`: closed-shell LB-B-Cl product complex, `uhf=0`
- `Cl_r`: closed-shell chlorinated substrate, `uhf=0`
- `Cl_p`: open-shell carbon radical product, `uhf=1`

Each species is written as `descriptor_inputs/<category>/<db_key>/input.xyz` with a matching `metadata.json`.

In [6]:
from pathlib import Path
import json
import os
import shutil
import sys

import pandas as pd

try:
    from ase.db import connect
    from ase.io import write
except ImportError as exc:
    raise ImportError(
        "This notebook needs ASE. Install the project environment first, for example: "
        "pip install ase pandas rdkit morfeus-ml"
    ) from exc

try:
    from DFTStructureGenerator import mol_manipulation
except ImportError:
    mol_manipulation = None

REPO_ROOT = Path.cwd()
DB_PATH = REPO_ROOT / "BorylXAT-DB.db"
TS_CSV_PATH = REPO_ROOT / "Data" / "TS" / "Borane_all.csv"
EXPORT_ROOT = Path(r"X:/BorylXAT/descriptor_inputs")

OVERWRITE_EXISTING = False
WRITE_RUN_SCRIPT = True

CATEGORY_UHF = {
    "B_N_r": 1,
    "B_N_p": 0,
    "Cl_r": 0,
    "Cl_p": 1,
}

SPIN_POPULATION_XCONTROL = """$write
   spin population=true
   json=true
$end
"""

for path in [DB_PATH, TS_CSV_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

print("Repository:", REPO_ROOT)
print("Database:", DB_PATH)
print("TS CSV:", TS_CSV_PATH)
print("Export root:", EXPORT_ROOT)

Repository: d:\OneDrive_all\OneDrive\Work\work_part\22.Borane_Radical_Database
Database: d:\OneDrive_all\OneDrive\Work\work_part\22.Borane_Radical_Database\BorylXAT-DB.db
TS CSV: d:\OneDrive_all\OneDrive\Work\work_part\22.Borane_Radical_Database\Data\TS\Borane_all.csv
Export root: X:\BorylXAT\descriptor_inputs


## Build Export Plan

The model combines one `B_N` descriptor block and one `Cl` descriptor block. We only export unique structures needed by `Borane_all.csv`.

In [2]:
def db_key_for(category, row):
    if category.startswith("B_N"):
        return f"B_{int(row['B_Index']):05}_LB_{int(row['N_Index']):05}_{category[-1]}"
    if category.startswith("Cl"):
        return f"Cl_{int(row['Cl_Index']):05}_{category[-1]}"
    raise ValueError(category)


def build_export_plan(ts_df):
    records = []

    bn_cols = ["B_Index", "N_Index"]
    bn_df = ts_df.sort_values(bn_cols).drop_duplicates(bn_cols).copy()
    for _, row in bn_df.iterrows():
        for category in ["B_N_r", "B_N_p"]:
            records.append({
                "category": category,
                "key": db_key_for(category, row),
                "B_Index": int(row["B_Index"]),
                "B_Atomid": int(row["B_Atomid"]),
                "N_Index": int(row["N_Index"]),
                "N_Atomid": int(row["N_Atomid"]),
                "B_smiles": row["B_smiles"],
                "N_smiles": row["N_smiles"],
                "Cl_Index": None,
                "Cl_Atomid": None,
                "Cl_smiles": None,
            })

    cl_cols = ["Cl_Index", "Cl_Atomid"]
    cl_df = ts_df.sort_values(cl_cols).drop_duplicates(cl_cols).copy()
    for _, row in cl_df.iterrows():
        for category in ["Cl_r", "Cl_p"]:
            records.append({
                "category": category,
                "key": db_key_for(category, row),
                "B_Index": None,
                "B_Atomid": None,
                "N_Index": None,
                "N_Atomid": None,
                "B_smiles": None,
                "N_smiles": None,
                "Cl_Index": int(row["Cl_Index"]),
                "Cl_Atomid": int(row["Cl_Atomid"]),
                "Cl_smiles": row["Cl_smiles"],
            })

    plan = pd.DataFrame(records).drop_duplicates(["category", "key"]).reset_index(drop=True)
    return plan


ts_df = pd.read_csv(TS_CSV_PATH)
plan_df = build_export_plan(ts_df)
print("TS rows:", len(ts_df))
print(plan_df.groupby("category").size().to_string())
plan_df.head()

TS rows: 9237
category
B_N_p    4363
B_N_r    4363
Cl_p      178
Cl_r      178


,category,key,B_Index,B_Atomid,N_Index,N_Atomid,B_smiles,N_smiles,Cl_Index,Cl_Atomid,Cl_smiles
0,B_N_r,B_00388_LB_00000_r,388.0,0.0,0.0,2.0,B,CCN(CC)CC,NaN,NaN,None
1,B_N_p,B_00388_LB_00000_p,388.0,0.0,0.0,2.0,B,CCN(CC)CC,NaN,NaN,None
2,B_N_r,B_00388_LB_00001_r,388.0,0.0,1.0,1.0,B,CN(C)C,NaN,NaN,None
3,B_N_p,B_00388_LB_00001_p,388.0,0.0,1.0,1.0,B,CN(C)C,NaN,NaN,None
4,B_N_r,B_00388_LB_00002_r,388.0,0.0,2.0,5.0,B,COC(OC)N(C)C,NaN,NaN,None


## Atom Mapping Helpers

The metadata stores 1-based atom indices because xTB output tables are 1-based. The original DFT descriptor code used the same atom ordering from the database structures.

In [3]:
def get_row_charge(db_row, default=0):
    for attr in ["charge"]:
        try:
            value = db_row.get(attr, None)
        except Exception:
            value = None
        if value is not None:
            try:
                return int(round(float(value)))
            except Exception:
                pass
    return int(default)


def mol_from_smiles(smiles):
    if mol_manipulation is None or smiles is None or pd.isna(smiles):
        return None
    return mol_manipulation.smiles2mol(smiles)


def atom_count_from_smiles(smiles):
    mol = mol_from_smiles(smiles)
    if mol is None:
        return None
    return int(mol.GetNumAtoms())


def first_symbol_index_1based(atoms, symbol):
    symbols = atoms.get_chemical_symbols()
    try:
        return symbols.index(symbol) + 1
    except ValueError:
        return None


def neighbor_index_for_cl_site_0based(cl_smiles, cl_atomid):
    mol = mol_from_smiles(cl_smiles)
    if mol is None:
        return None
    atom = mol.GetAtomWithIdx(int(cl_atomid))
    neighbors = list(atom.GetNeighbors())
    if not neighbors:
        return None
    return int(neighbors[0].GetIdx())


def jsonable(value):
    if value is None:
        return None
    try:
        if pd.isna(value):
            return None
    except Exception:
        pass
    if hasattr(value, "item"):
        return value.item()
    return value


def reaction_site_metadata(category, record, atoms):
    record = dict(record)
    site = {"index_base": 1}

    if category == "B_N_r":
        b_count = atom_count_from_smiles(record["B_smiles"])
        site["B_atom_1based"] = int(record["B_Atomid"]) + 1
        site["N_atom_1based"] = None if b_count is None else int(record["N_Atomid"]) + b_count
        site["distance_descriptor"] = ["B_atom_1based", "N_atom_1based"]
        site["spin_descriptor_atom"] = "B_atom_1based"
        site["charge_descriptor_atoms"] = ["B_atom_1based"]
        site["orbital_descriptor"] = "SOMO_from_highest_occupied_orbital"

    elif category == "B_N_p":
        site["B_atom_1based"] = int(record["B_Atomid"]) + 1
        site["Cl_atom_1based"] = first_symbol_index_1based(atoms, "Cl")
        site["distance_descriptor"] = ["B_atom_1based", "Cl_atom_1based"]
        site["charge_descriptor_atoms"] = ["B_atom_1based", "Cl_atom_1based"]
        site["buried_volume"] = {"center": "B_atom_1based", "axis": "Cl_atom_1based", "radius_A": 6.0}
        site["orbital_descriptor"] = "LUMO"

    elif category == "Cl_r":
        c_idx = neighbor_index_for_cl_site_0based(record["Cl_smiles"], record["Cl_Atomid"])
        site["Cl_atom_1based"] = int(record["Cl_Atomid"]) + 1
        site["C_atom_1based"] = None if c_idx is None else c_idx + 1
        site["distance_descriptor"] = ["Cl_atom_1based", "C_atom_1based"]
        site["charge_descriptor_atoms"] = ["Cl_atom_1based", "C_atom_1based"]
        site["buried_volume"] = {"center": "C_atom_1based", "axis": "Cl_atom_1based", "radius_A": 6.0}
        site["orbital_descriptor"] = "LUMO"

    elif category == "Cl_p":
        c_idx = neighbor_index_for_cl_site_0based(record["Cl_smiles"], record["Cl_Atomid"])
        if c_idx is not None and c_idx > int(record["Cl_Atomid"]):
            c_idx -= 1
        site["C_radical_atom_1based"] = None if c_idx is None else c_idx + 1
        site["spin_descriptor_atom"] = "C_radical_atom_1based"
        site["charge_descriptor_atoms"] = ["C_radical_atom_1based"]
        site["orbital_descriptor"] = "SOMO_from_highest_occupied_orbital"

    return site

## Export Structures

This cell writes one folder per database key. Re-run with `OVERWRITE_EXISTING=True` if you intentionally want to replace old inputs.

In [4]:
def write_run_script(folder, charge, uhf):
    script = f"""#!/usr/bin/env bash
set -euo pipefail
cd "$(dirname "$0")"

export OMP_NUM_THREADS="${{OMP_NUM_THREADS:-16}}"
export MKL_NUM_THREADS="${{MKL_NUM_THREADS:-$OMP_NUM_THREADS}}"
export OMP_STACKSIZE="${{OMP_STACKSIZE:-4G}}"

xtb input.xyz --gfn 2 --opt tight --chrg {charge} --uhf {uhf} --norestart > opt.out

if [ {uhf} -eq 1 ]; then
  xtb xtbopt.xyz --gfn 2 --chrg {charge} --uhf {uhf} -I spin_population.inp --pop --json --norestart > sp.out
else
  xtb xtbopt.xyz --gfn 2 --chrg {charge} --uhf {uhf} --pop --json --norestart > sp.out
fi

if [ "${{RUN_OHESS:-0}}" = "1" ]; then
  xtb xtbopt.xyz --gfn 2 --ohess tight --chrg {charge} --uhf {uhf} --norestart > ohess.out
fi
"""
    path = folder / "run_xtb.sh"
    path.write_text(script, encoding="utf-8", newline="\n")
    try:
        path.chmod(0o755)
    except Exception:
        pass


def export_one(db, record):
    category = record["category"]
    key = record["key"]
    folder = EXPORT_ROOT / category / key
    xyz_path = folder / "input.xyz"
    metadata_path = folder / "metadata.json"

    if folder.exists() and not OVERWRITE_EXISTING and xyz_path.exists() and metadata_path.exists():
        return {"category": category, "key": key, "status": "skipped_exists", "folder": str(folder)}

    folder.mkdir(parents=True, exist_ok=True)
    db_row = db.get(key=key)
    atoms = db_row.toatoms()
    charge = get_row_charge(db_row, default=0)
    uhf = CATEGORY_UHF[category]
    site = reaction_site_metadata(category, record, atoms)

    write(xyz_path, atoms, format="xyz")
    (folder / "spin_population.inp").write_text(SPIN_POPULATION_XCONTROL, encoding="utf-8", newline="\n")

    metadata = {
        "key": key,
        "db_key": key,
        "category": category,
        "source": "BorylXAT-DB.db",
        "source_csv": str(TS_CSV_PATH),
        "charge": charge,
        "uhf": uhf,
        "n_atoms": len(atoms),
        "formula": atoms.get_chemical_formula(),
        "reaction_site": site,
        "csv_record": {k: jsonable(v) for k, v in dict(record).items()},
        "descriptor_role": {
            "B_N_r": "LB-B radical complex descriptors",
            "B_N_p": "LB-B-Cl product complex descriptors",
            "Cl_r": "chlorinated substrate descriptors",
            "Cl_p": "carbon radical product descriptors",
        }[category],
    }
    metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    if WRITE_RUN_SCRIPT:
        write_run_script(folder, charge, uhf)

    return {"category": category, "key": key, "status": "exported", "folder": str(folder)}


EXPORT_ROOT.mkdir(parents=True, exist_ok=True)
plan_df.to_csv(EXPORT_ROOT / "xtb_descriptor_export_plan.csv", index=False)

results = []
db = connect(DB_PATH)
for record in plan_df.to_dict("records"):
    try:
        results.append(export_one(db, record))
    except Exception as exc:
        results.append({
            "category": record["category"],
            "key": record["key"],
            "status": "error",
            "error": repr(exc),
            "folder": str(EXPORT_ROOT / record["category"] / record["key"]),
        })

manifest_df = pd.DataFrame(results)
manifest_df.to_csv(EXPORT_ROOT / "export_manifest.csv", index=False)
print(manifest_df.groupby(["category", "status"]).size().to_string())
manifest_df.head()

category  status  
B_N_p     exported    4363
B_N_r     exported    4363
Cl_p      exported     178
Cl_r      exported     178


,category,key,status,folder
0,B_N_r,B_00388_LB_00000_r,exported,X:\BorylXAT\descriptor_inputs\B_N_r\B_00388_LB...
1,B_N_p,B_00388_LB_00000_p,exported,X:\BorylXAT\descriptor_inputs\B_N_p\B_00388_LB...
2,B_N_r,B_00388_LB_00001_r,exported,X:\BorylXAT\descriptor_inputs\B_N_r\B_00388_LB...
3,B_N_p,B_00388_LB_00001_p,exported,X:\BorylXAT\descriptor_inputs\B_N_p\B_00388_LB...
4,B_N_r,B_00388_LB_00002_r,exported,X:\BorylXAT\descriptor_inputs\B_N_r\B_00388_LB...


## Validate Export Tree

A quick summary of folder counts and any missing/error entries.

In [5]:
manifest_path = EXPORT_ROOT / "export_manifest.csv"
manifest_df = pd.read_csv(manifest_path)
print("Manifest:", manifest_path)
print(manifest_df.groupby(["category", "status"]).size().to_string())

errors = manifest_df[manifest_df["status"] == "error"]
if len(errors):
    display(errors.head(20))
else:
    print("No export errors recorded.")

for category in CATEGORY_UHF:
    category_dir = EXPORT_ROOT / category
    n_folders = len([p for p in category_dir.iterdir() if p.is_dir()]) if category_dir.exists() else 0
    print(f"{category}: {n_folders} folders")

Manifest: X:\BorylXAT\descriptor_inputs\export_manifest.csv
category  status  
B_N_p     exported    4363
B_N_r     exported    4363
Cl_p      exported     178
Cl_r      exported     178
No export errors recorded.
B_N_r: 4363 folders
B_N_p: 4363 folders
Cl_r: 178 folders
Cl_p: 178 folders


## Build xTB Descriptor Dictionaries

This section converts completed xTB outputs under `X:/BorylXAT/descriptor_inputs` into descriptor maps with the same structure as the original DFT pickle files:

- `BNdes_xtb.pkl`: keys like `B_00388_Nu_00000`, values of length 10
- `Cldes_xtb.pkl`: keys like `Cl_00500`, values of length 9

The feature order follows `DFTStructureGenerator.descriptor` exactly, so the existing `dataframe_to_descriptors(...)` function can be reused after swapping the descriptor maps.

In [7]:
import json
import math
import pickle
import re
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from morfeus import BuriedVolume
except ImportError as exc:
    raise ImportError(
        "This descriptor-map build needs morfeus for %Vbur. Install with: pip install morfeus-ml"
    ) from exc

# Keep this copy self-contained so the notebook can build xTB maps even before importing descriptor.py.
DUPLICATE_CL_IDS = [
    624, 625, 626, 627, 628, 629, 630, 631, 632, 633, 634, 635, 636,
    637, 638, 639, 640, 642, 644, 645, 652, 653, 654, 655, 656, 657,
    658, 659, 660, 661, 662, 663, 664, 665, 666, 667, 668, 669, 670,
    671, 672, 673, 674, 675, 676, 677, 678, 679, 680, 681, 682, 683,
    684, 685, 686, 687, 688, 689, 690, 691, 692, 693, 694, 695, 696,
    697, 698, 699, 700, 701, 702, 703, 704, 705, 706, 707, 708, 709,
    710, 711, 713, 714, 716, 717, 718, 719, 720, 721, 722,
]

XTB_ROOT = Path(r"X:/BorylXAT")
XTB_INPUT_ROOT = XTB_ROOT / "descriptor_inputs"
XTB_MAP_DIR = XTB_ROOT / "descriptor_maps"
LOCAL_DESCRIPTOR_DIR = REPO_ROOT / "Data" / "descriptor"

BN_XTB_DESCRIPTOR_PATH = LOCAL_DESCRIPTOR_DIR / "BNdes_xtb.pkl"
CL_XTB_DESCRIPTOR_PATH = LOCAL_DESCRIPTOR_DIR / "Cldes_xtb.pkl"
BN_XTB_DESCRIPTOR_MIRROR = XTB_MAP_DIR / "BNdes_xtb.pkl"
CL_XTB_DESCRIPTOR_MIRROR = XTB_MAP_DIR / "Cldes_xtb.pkl"
DIAGNOSTICS_PATH = XTB_ROOT / "xtb_descriptor_map_diagnostics.csv"
SUMMARY_PATH = XTB_ROOT / "xtb_descriptor_map_summary.csv"

# For cheaper xTB variants this is usually available for every SP output.
# If every species has ohess.out, set ENERGY_SOURCE = "total_free_energy_Eh".
ENERGY_SOURCE = "total_energy_Eh"

# DFT descriptors store orbital energies in kcal/mol. xTB JSON gives eV.
ORBITAL_OUTPUT_UNIT = "kcal/mol"  # alternatives: "eV"
EV_TO_KCAL_MOL = 23.06054783061903

for path in [XTB_INPUT_ROOT, LOCAL_DESCRIPTOR_DIR]:
    path.mkdir(parents=True, exist_ok=True)
XTB_MAP_DIR.mkdir(parents=True, exist_ok=True)

print("xTB input root:", XTB_INPUT_ROOT)
print("Energy source:", ENERGY_SOURCE)
print("Orbital unit:", ORBITAL_OUTPUT_UNIT)

xTB input root: X:\BorylXAT\descriptor_inputs
Energy source: total_energy_Eh
Orbital unit: kcal/mol


### Parsers

The parser reads `metadata.json`, `xtbout.json`, `xtbopt.xyz`, `sp.out`, and optional `ohess.out`. Atom indices in metadata are 1-based, matching xTB output tables.

In [8]:
def read_json(path):
    path = Path(path)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def read_json_if_exists(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size == 0:
        return None
    return read_json(path)


def json_get(data, *names):
    if data is None:
        return None
    for name in names:
        if name in data:
            return data[name]
    return None


def parse_xyz(path):
    lines = Path(path).read_text(encoding="utf-8", errors="ignore").splitlines()
    n_atoms = int(lines[0].strip())
    symbols = []
    positions = []
    for line in lines[2:2 + n_atoms]:
        parts = line.split()
        symbols.append(parts[0])
        positions.append([float(parts[1]), float(parts[2]), float(parts[3])])
    return symbols, np.asarray(positions, dtype=float)


def distance_from_positions(positions, atom_i_1based, atom_j_1based):
    if atom_i_1based is None or atom_j_1based is None:
        raise ValueError("Missing atom index for distance descriptor")
    i = int(atom_i_1based) - 1
    j = int(atom_j_1based) - 1
    return float(np.linalg.norm(positions[i] - positions[j]))


def parse_spin_population(sp_out_path):
    path = Path(sp_out_path)
    if not path.exists() or path.stat().st_size == 0:
        return {}
    spin = {}
    inside = False
    for line in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        if "(R)spin-density population" in line:
            inside = True
            continue
        if inside and line.startswith("Wiberg/"):
            break
        match = re.match(r"\s*(\d+)([A-Za-z]+)\s+([-+]?\d+\.\d+)", line)
        if match:
            spin[int(match.group(1))] = float(match.group(3))
    return spin


def parse_ohess(ohess_path):
    path = Path(ohess_path)
    if not path.exists() or path.stat().st_size == 0:
        return {}
    text = path.read_text(encoding="utf-8", errors="ignore")
    def grab(pattern):
        match = re.search(pattern, text)
        return float(match.group(1)) if match else None
    def grab_int(pattern):
        match = re.search(pattern, text)
        return int(match.group(1)) if match else None
    return {
        "total_free_energy_Eh": grab(r"total free energy\s+([-+]?\d+\.\d+) Eh"),
        "zero_point_energy_Eh": grab(r"zero point energy\s+([-+]?\d+\.\d+) Eh"),
        "imaginary_frequencies": grab_int(r"# imaginary freq\.\s+(\d+)"),
    }


def convert_orbital_energy(value_eV):
    if value_eV is None:
        return None
    value_eV = float(value_eV)
    if ORBITAL_OUTPUT_UNIT == "eV":
        return value_eV
    if ORBITAL_OUTPUT_UNIT == "kcal/mol":
        return value_eV * EV_TO_KCAL_MOL
    raise ValueError(f"Unsupported ORBITAL_OUTPUT_UNIT: {ORBITAL_OUTPUT_UNIT}")


def frontier_orbitals_from_json(data):
    occ = json_get(data, "fractional occupation")
    orbital_e = json_get(data, "orbital energies / eV", "orbital energies/eV")
    if occ is None or orbital_e is None:
        raise ValueError("Missing fractional occupation or orbital energies in xtbout.json")
    occupied = [idx for idx, value in enumerate(occ) if float(value) > 1e-6]
    unoccupied = [idx for idx, value in enumerate(occ) if float(value) < 1e-6]
    if not occupied or not unoccupied:
        raise ValueError("Could not identify occupied/unoccupied frontier orbitals")
    homo_idx = occupied[-1]
    lumo_idx = unoccupied[0]
    return {
        "homo_or_somo_e": convert_orbital_energy(orbital_e[homo_idx]),
        "lumo_e": convert_orbital_energy(orbital_e[lumo_idx]),
        "homo_or_somo_index_1based": homo_idx + 1,
        "lumo_index_1based": lumo_idx + 1,
    }


def sum_buried_volume_from_1based(symbols, positions, center_atom_1based, axis_atom_1based, radius=6.0):
    if center_atom_1based is None or axis_atom_1based is None:
        raise ValueError("Missing atom index for buried volume descriptor")
    bv = BuriedVolume(
        symbols,
        positions,
        int(center_atom_1based),
        include_hs=True,
        radius=float(radius),
        z_axis_atoms=[int(axis_atom_1based)],
        excluded_atoms=[int(axis_atom_1based)],
    )
    bv.octant_analysis()
    return float(np.sum(list(bv.octants["percent_buried_volume"].values())))


def cl_descriptor_name(cl_index, cl_atomid):
    cl_index = int(cl_index)
    if cl_index in DUPLICATE_CL_IDS:
        return f"Cl_{cl_index:05}_Claid_{int(cl_atomid):05}"
    return f"Cl_{cl_index:05}"

### Load Completed Species Outputs

Each completed folder is converted into a species record. The later cells combine paired records into `BNdes` and `Cldes` maps.

In [9]:
def load_species_record(folder):
    folder = Path(folder)
    meta = read_json(folder / "metadata.json")
    data = read_json_if_exists(folder / "xtbout.json")
    xyz_path = folder / "xtbopt.xyz"
    if data is None:
        raise FileNotFoundError(folder / "xtbout.json")
    if not xyz_path.exists():
        raise FileNotFoundError(xyz_path)

    symbols, positions = parse_xyz(xyz_path)
    charges = json_get(data, "partial charges")
    if charges is None:
        raise ValueError("Missing partial charges in xtbout.json")
    frontier = frontier_orbitals_from_json(data)
    spin = parse_spin_population(folder / "sp.out")
    thermo = parse_ohess(folder / "ohess.out")

    record = {
        "key": meta["key"],
        "category": meta["category"],
        "folder": str(folder),
        "metadata": meta,
        "symbols": symbols,
        "positions": positions,
        "charges": [float(x) for x in charges],
        "spin_population": spin,
        "total_energy_Eh": json_get(data, "total energy"),
        "electronic_energy_Eh": json_get(data, "electronic energy"),
        **frontier,
        **thermo,
    }
    for field in ["total_energy_Eh", "electronic_energy_Eh", "total_free_energy_Eh"]:
        if record.get(field) is not None:
            record[field] = float(record[field])
    return record


def load_category_records(category):
    records = {}
    errors = []
    category_dir = XTB_INPUT_ROOT / category
    for metadata_path in sorted(category_dir.glob("*/metadata.json")):
        folder = metadata_path.parent
        try:
            rec = load_species_record(folder)
            records[rec["key"]] = rec
        except Exception as exc:
            errors.append({
                "category": category,
                "key": folder.name,
                "folder": str(folder),
                "status": "species_load_error",
                "error": repr(exc),
            })
    return records, errors

species_records = {}
species_errors = []
for category in ["B_N_r", "B_N_p", "Cl_r", "Cl_p"]:
    records, errors = load_category_records(category)
    species_records[category] = records
    species_errors.extend(errors)
    print(category, "loaded", len(records), "errors", len(errors))

pd.DataFrame(species_errors).head()

B_N_r loaded 4363 errors 0
B_N_p loaded 4363 errors 0
Cl_r loaded 178 errors 0
Cl_p loaded 178 errors 0


""


### Build BN and Cl Descriptor Maps

Feature order is intentionally identical to the original DFT maps.

In [10]:
def energy_value(record):
    value = record.get(ENERGY_SOURCE)
    if value is None:
        raise ValueError(f"Missing requested energy source {ENERGY_SOURCE} for {record['key']}")
    return float(value)


def site_value(record, name):
    site = record["metadata"].get("reaction_site", {})
    value = site.get(name)
    if value is None:
        raise ValueError(f"Missing reaction_site.{name} for {record['key']}")
    return int(value)


def charge_at(record, atom_1based):
    return float(record["charges"][int(atom_1based) - 1])


def spin_at(record, atom_1based):
    spin = record.get("spin_population", {})
    atom_1based = int(atom_1based)
    if atom_1based not in spin:
        raise ValueError(f"Missing spin population for atom {atom_1based} in {record['key']}")
    return float(spin[atom_1based])


def bn_map_name_from_record(record):
    csv_record = record["metadata"].get("csv_record", {})
    return f"B_{int(csv_record['B_Index']):05}_Nu_{int(csv_record['N_Index']):05}"


def cl_map_name_from_record(record):
    csv_record = record["metadata"].get("csv_record", {})
    return cl_descriptor_name(csv_record["Cl_Index"], csv_record["Cl_Atomid"])


def build_one_bn_descriptor(react_record, prod_record):
    b_r = site_value(react_record, "B_atom_1based")
    n_r = site_value(react_record, "N_atom_1based")
    b_p = site_value(prod_record, "B_atom_1based")
    cl_p = site_value(prod_record, "Cl_atom_1based")

    reaction_energy = energy_value(prod_record) - energy_value(react_record)
    descriptor = [reaction_energy]
    descriptor += [spin_at(react_record, b_r)]
    descriptor += [charge_at(react_record, b_r)]
    descriptor += [distance_from_positions(react_record["positions"], b_r, n_r)]
    descriptor += [float(react_record["homo_or_somo_e"])]
    descriptor += [charge_at(prod_record, b_p), charge_at(prod_record, cl_p)]
    descriptor += [distance_from_positions(prod_record["positions"], b_p, cl_p)]
    descriptor += [float(prod_record["lumo_e"])]
    descriptor += [
        sum_buried_volume_from_1based(
            prod_record["symbols"],
            prod_record["positions"],
            b_p,
            cl_p,
            radius=6.0,
        )
    ]
    return [float(x) for x in descriptor]


def build_one_cl_descriptor(react_record, prod_record):
    cl_r = site_value(react_record, "Cl_atom_1based")
    c_r = site_value(react_record, "C_atom_1based")
    c_p = site_value(prod_record, "C_radical_atom_1based")

    reaction_energy = energy_value(prod_record) - energy_value(react_record)
    descriptor = [reaction_energy]
    descriptor += [charge_at(react_record, cl_r), charge_at(react_record, c_r)]
    descriptor += [distance_from_positions(react_record["positions"], cl_r, c_r)]
    descriptor += [float(react_record["lumo_e"])]
    descriptor += [
        sum_buried_volume_from_1based(
            react_record["symbols"],
            react_record["positions"],
            c_r,
            cl_r,
            radius=6.0,
        )
    ]
    descriptor += [spin_at(prod_record, c_p)]
    descriptor += [charge_at(prod_record, c_p)]
    descriptor += [float(prod_record["homo_or_somo_e"])]
    return [float(x) for x in descriptor]


def build_xtb_descriptor_maps():
    bn_map = {}
    cl_map = {}
    diagnostics = []

    for react_key, react_record in sorted(species_records["B_N_r"].items()):
        prod_key = react_key[:-2] + "_p"
        map_name = bn_map_name_from_record(react_record)
        prod_record = species_records["B_N_p"].get(prod_key)
        if prod_record is None:
            diagnostics.append({"map": "BN", "name": map_name, "status": "missing_product", "react_key": react_key, "prod_key": prod_key})
            continue
        try:
            bn_map[map_name] = build_one_bn_descriptor(react_record, prod_record)
            diagnostics.append({"map": "BN", "name": map_name, "status": "ok", "react_key": react_key, "prod_key": prod_key})
        except Exception as exc:
            diagnostics.append({"map": "BN", "name": map_name, "status": "descriptor_error", "react_key": react_key, "prod_key": prod_key, "error": repr(exc)})

    for react_key, react_record in sorted(species_records["Cl_r"].items()):
        prod_key = react_key[:-2] + "_p"
        map_name = cl_map_name_from_record(react_record)
        prod_record = species_records["Cl_p"].get(prod_key)
        if prod_record is None:
            diagnostics.append({"map": "Cl", "name": map_name, "status": "missing_product", "react_key": react_key, "prod_key": prod_key})
            continue
        try:
            cl_map[map_name] = build_one_cl_descriptor(react_record, prod_record)
            diagnostics.append({"map": "Cl", "name": map_name, "status": "ok", "react_key": react_key, "prod_key": prod_key})
        except Exception as exc:
            diagnostics.append({"map": "Cl", "name": map_name, "status": "descriptor_error", "react_key": react_key, "prod_key": prod_key, "error": repr(exc)})

    return bn_map, cl_map, pd.DataFrame(diagnostics)

B_N_des_map_xtb, Cl_des_map_xtb, descriptor_diagnostics_df = build_xtb_descriptor_maps()
print("BN descriptors:", len(B_N_des_map_xtb))
print("Cl descriptors:", len(Cl_des_map_xtb))
print(descriptor_diagnostics_df.groupby(["map", "status"]).size().to_string())
display(descriptor_diagnostics_df.head())

BN descriptors: 4363
Cl descriptors: 178
map  status
BN   ok        4363
Cl   ok         178


,map,name,status,react_key,prod_key
0,BN,B_00388_Nu_00000,ok,B_00388_LB_00000_r,B_00388_LB_00000_p
1,BN,B_00388_Nu_00001,ok,B_00388_LB_00001_r,B_00388_LB_00001_p
2,BN,B_00388_Nu_00002,ok,B_00388_LB_00002_r,B_00388_LB_00002_p
3,BN,B_00388_Nu_00003,ok,B_00388_LB_00003_r,B_00388_LB_00003_p
4,BN,B_00388_Nu_00006,ok,B_00388_LB_00006_r,B_00388_LB_00006_p


### Save xTB Descriptor Maps

The local files under `Data/descriptor` are the ones to use from modeling notebooks. A mirror is also written to `X:/BorylXAT/descriptor_maps` for provenance next to the xTB outputs.

In [11]:
with open(BN_XTB_DESCRIPTOR_PATH, "wb") as f:
    pickle.dump(B_N_des_map_xtb, f)
with open(CL_XTB_DESCRIPTOR_PATH, "wb") as f:
    pickle.dump(Cl_des_map_xtb, f)
with open(BN_XTB_DESCRIPTOR_MIRROR, "wb") as f:
    pickle.dump(B_N_des_map_xtb, f)
with open(CL_XTB_DESCRIPTOR_MIRROR, "wb") as f:
    pickle.dump(Cl_des_map_xtb, f)

diagnostics_full_df = pd.concat([
    descriptor_diagnostics_df,
    pd.DataFrame(species_errors),
], ignore_index=True, sort=False)
diagnostics_full_df.to_csv(DIAGNOSTICS_PATH, index=False)

summary_rows = [
    {"item": "BN_map_entries", "value": len(B_N_des_map_xtb)},
    {"item": "Cl_map_entries", "value": len(Cl_des_map_xtb)},
    {"item": "BN_descriptor_length", "value": len(next(iter(B_N_des_map_xtb.values()))) if B_N_des_map_xtb else None},
    {"item": "Cl_descriptor_length", "value": len(next(iter(Cl_des_map_xtb.values()))) if Cl_des_map_xtb else None},
    {"item": "energy_source", "value": ENERGY_SOURCE},
    {"item": "orbital_unit", "value": ORBITAL_OUTPUT_UNIT},
    {"item": "BN_pickle", "value": str(BN_XTB_DESCRIPTOR_PATH)},
    {"item": "Cl_pickle", "value": str(CL_XTB_DESCRIPTOR_PATH)},
    {"item": "diagnostics_csv", "value": str(DIAGNOSTICS_PATH)},
]
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_PATH, index=False)

print("Saved:")
print(" ", BN_XTB_DESCRIPTOR_PATH)
print(" ", CL_XTB_DESCRIPTOR_PATH)
print(" ", BN_XTB_DESCRIPTOR_MIRROR)
print(" ", CL_XTB_DESCRIPTOR_MIRROR)
print("Diagnostics:", DIAGNOSTICS_PATH)
display(summary_df)

Saved:
  d:\OneDrive_all\OneDrive\Work\work_part\22.Borane_Radical_Database\Data\descriptor\BNdes_xtb.pkl
  d:\OneDrive_all\OneDrive\Work\work_part\22.Borane_Radical_Database\Data\descriptor\Cldes_xtb.pkl
  X:\BorylXAT\descriptor_maps\BNdes_xtb.pkl
  X:\BorylXAT\descriptor_maps\Cldes_xtb.pkl
Diagnostics: X:\BorylXAT\xtb_descriptor_map_diagnostics.csv


,item,value
0,BN_map_entries,4363
1,Cl_map_entries,178
2,BN_descriptor_length,10
3,Cl_descriptor_length,9
4,energy_source,total_energy_Eh
5,orbital_unit,kcal/mol
6,BN_pickle,d:\OneDrive_all\OneDrive\Work\work_part\22.Bor...
7,Cl_pickle,d:\OneDrive_all\OneDrive\Work\work_part\22.Bor...
8,diagnostics_csv,X:\BorylXAT\xtb_descriptor_map_diagnostics.csv


### Compatibility Check Against `Borane_all.csv`

This check verifies whether the current xTB maps cover every reaction in `Data/TS/Borane_all.csv`. If coverage is incomplete, inspect `xtb_descriptor_map_diagnostics.csv` and continue xTB jobs before training the full xTB model.

In [12]:
def required_descriptor_names_from_ts(ts_df):
    required_bn = {
        f"B_{int(row.B_Index):05}_Nu_{int(row.N_Index):05}"
        for row in ts_df.itertuples(index=False)
    }
    required_cl = {
        cl_descriptor_name(int(row.Cl_Index), int(row.Cl_Atomid))
        for row in ts_df.itertuples(index=False)
    }
    return required_bn, required_cl

required_bn, required_cl = required_descriptor_names_from_ts(pd.read_csv(TS_CSV_PATH))
missing_bn = sorted(required_bn - set(B_N_des_map_xtb))
missing_cl = sorted(required_cl - set(Cl_des_map_xtb))

coverage_df = pd.DataFrame([
    {"map": "BN", "required": len(required_bn), "available": len(B_N_des_map_xtb), "missing": len(missing_bn)},
    {"map": "Cl", "required": len(required_cl), "available": len(Cl_des_map_xtb), "missing": len(missing_cl)},
])
display(coverage_df)

if missing_bn:
    print("First missing BN keys:", missing_bn[:20])
if missing_cl:
    print("First missing Cl keys:", missing_cl[:20])

if not missing_bn and not missing_cl:
    print("Coverage complete. These pickle files can be used as drop-in descriptor maps for dataframe_to_descriptors.")
else:
    print("Coverage incomplete. The partial pickle files were still saved for inspection/debugging.")

,map,required,available,missing
0,BN,4363,4363,0
1,Cl,178,178,0


Coverage complete. These pickle files can be used as drop-in descriptor maps for dataframe_to_descriptors.
